# Digital control: notebook

The running example of the Note: the chapter's robot (mass 8 kg, rolling drag 2 N s/m, drive force limited to ±4 N) and its PI speed controller (kp = 20 N per m/s, ki = 10 N per m, the gains of RO-060) run as code, once every sample period T. The robot cruises at 0.4 m/s and the command steps to 0.5 m/s at t = 0. Every number in the Note is computed here. Nothing is random, so no seed is needed.

In [1]:
import numpy as np
from scipy import signal

M, B, U_MAX = 8.0, 2.0, 4.0        # kg, N s/m, N
KP, KI = 20.0, 10.0                # N per m/s, N per m
DTP = 0.0005                       # s: the fine step that stands for continuous time
R0, R1 = 0.4, 0.5                  # m/s: speed before and after the command step
W_RIP, A_RIP = 10.0, 0.01          # rad/s and m/s: the wheel's once-per-turn ripple in the reading


def sampled_loop(fs, method="fwd", T=8.0, ripple=0.0, tau_f=None, r0=R0, r1=R1, digits=None):
    """PI speed loop run as code at fs Hz; the plant between ticks is stepped every 0.5 ms.
    method: 'fwd' (forward Euler: output first, then I += ki T e), 'bwd' (backward Euler) or 'tus' (Tustin)."""
    n_hold = int(round(1.0 / (fs * DTP)))
    Ts = n_hold * DTP
    n = int(round(T / DTP))
    v, I, u, vf, e_last = r0, B * r0, B * r0, r0, 0.0
    t = np.arange(n + 1) * DTP
    vs = np.empty(n + 1)
    rows = []
    for i in range(n + 1):
        y = v + ripple * np.sin(W_RIP * t[i])           # what the sensor reads
        if tau_f:
            vf += DTP * (y - vf) / tau_f                 # analog low-pass filter before the A/D
            y = vf
        if i % n_hold == 0:                              # clock tick: A/D, compute, D/A
            if digits is not None:
                y = round(y, digits)
            e = r1 - y
            if method == "fwd":
                c = KP * e + I
                I_next = I + KI * Ts * e
            elif method == "bwd":
                I_next = I + KI * Ts * e
                c = KP * e + I_next
            else:
                I_next = I + KI * Ts * (e + e_last) / 2
                c = KP * e + I_next
            u = float(np.clip(c, -U_MAX, U_MAX))
            rows.append((t[i], y, e, u, I))
            I, e_last = I_next, e
        vs[i] = v
        v += DTP * (u - B * v) / M
    k = np.array(rows)
    return dict(t=t, v=vs, tk=k[:, 0], yk=k[:, 1], ek=k[:, 2], uk=k[:, 3], ik=k[:, 4])


def overshoot(r, r0=R0, r1=R1):
    return (r["v"].max() - r1) / (r1 - r0) * 100

## 1. The opening problem: the same controller at 100, 10 and 2 Hz (Note, Section 1)

In [2]:
cont = sampled_loop(1 / DTP)
print("continuous formula: overshoot", round(overshoot(cont), 1), "percent")
for fs in (100, 10, 2, 1):
    print(f"{fs:>4} Hz: overshoot {overshoot(sampled_loop(fs)):.1f} percent")
r = sampled_loop(0.5, T=60)
late = r["v"][r["t"] > 20]
print("0.5 Hz: after 20 s the speed still swings between", round(late.min(), 2), "and", round(late.max(), 2), "m/s")

continuous formula: overshoot 4.9 percent
 100 Hz: overshoot 5.0 percent
  10 Hz: overshoot 5.7 percent
   2 Hz: overshoot 17.5 percent
   1 Hz: overshoot 121.2 percent
0.5 Hz: after 20 s the speed still swings between -0.21 and 1.2 m/s


## 2. The zero-order hold delays by half a sample (Note, Section 3.2)

The continuous loop with a pure delay of 0.25 s on its force, against the loop run at 2 Hz.

In [3]:
def delayed_loop(D, T=10.0):
    nd = int(round(D / DTP))
    v, I = R0, B * R0
    hist = [B * R0] * (nd + 1)
    vs = []
    for i in range(int(T / DTP)):
        e = R1 - v
        hist.append(KP * e + I)
        I += KI * DTP * e
        v += DTP * (hist[-1 - nd] - B * v) / M
        vs.append(v)
    return max(vs)

for D, fs in ((0.05, 10), (0.25, 2)):
    print(f"continuous loop + {D} s delay: overshoot {(delayed_loop(D) - R1) / 0.1 * 100:.1f} percent;",
          f"loop run at {fs} Hz: {overshoot(sampled_loop(fs)):.1f} percent")
# phase margin of the continuous loop and what half a sample of delay takes away
L = signal.TransferFunction([KP, KI], [M, B, 0])
w = np.logspace(-2, 2, 400001)
H = signal.freqresp(L, w)[1]
i = np.argmin(abs(abs(H) - 1))
wc, pm = w[i], 180 + np.degrees(np.angle(H[i]))
print("crossover", round(wc, 3), "rad/s, phase margin", round(pm, 1), "degrees")
for fs in (100, 10, 2):
    lost = np.degrees(wc / fs / 2)
    print(f"{fs:>4} Hz: half a sample {1 / fs / 2:.3f} s costs {lost:.1f} degrees, leaving {round(pm, 1) - round(lost, 1):.1f}")

continuous loop + 0.05 s delay: overshoot 5.4 percent; loop run at 10 Hz: 5.7 percent


continuous loop + 0.25 s delay: overshoot 23.9 percent; loop run at 2 Hz: 17.5 percent
crossover 2.536 rad/s, phase margin 84.5 degrees
 100 Hz: half a sample 0.005 s costs 0.7 degrees, leaving 83.8
  10 Hz: half a sample 0.050 s costs 7.3 degrees, leaving 77.2
   2 Hz: half a sample 0.250 s costs 36.3 degrees, leaving 48.2


## 3. The PI controller as code: the first ticks at 10 Hz (Note, Section 4.2)

The A/D rounds each reading to 0.0001 m/s, so every number below is exact.

In [4]:
r = sampled_loop(10, T=1.1, digits=4)
for k in range(4):
    print(f"tick {k}: v = {r['yk'][k]:.4f}, e = {r['ek'][k]:.4f}, u = {r['uk'][k]:.5f}, I before = {r['ik'][k]:.5f}, "
          f"I after = {r['ik'][k] + KI * 0.1 * r['ek'][k]:.5f}")

tick 0: v = 0.4000, e = 0.1000, u = 2.80000, I before = 0.80000, I after = 0.90000
tick 1: v = 0.4247, e = 0.0753, u = 2.40600, I before = 0.90000, I after = 0.97530
tick 2: v = 0.4439, e = 0.0561, u = 2.09730, I before = 0.97530, I after = 1.03140
tick 3: v = 0.4588, e = 0.0412, u = 1.85540, I before = 1.03140, I after = 1.07260


## 4. Three rules for the area under the error (Note, Section 4.3)

In [5]:
T, N = 0.5, 5
t, e = cont["t"], R1 - cont["v"]
tk = np.arange(N + 1) * T
ek = np.interp(tk, t, e)
m = t <= N * T + 1e-9
print("error samples", np.round(ek, 4))
print("exact area    ", round(np.trapezoid(e[m], t[m]), 4), "m")
print("forward Euler ", round(T * ek[:-1].sum(), 4))
print("backward Euler", round(T * ek[1:].sum(), 4))
print("Tustin        ", round(T * (ek[:-1] + ek[1:]).sum() / 2, 4))

error samples [ 0.1     0.0253  0.0022 -0.004  -0.0049 -0.0043]
exact area     0.0281 m
forward Euler  0.0593
backward Euler 0.0072
Tustin         0.0333


## 5. Which rule stays stable: a filter with time constant 0.02 s at 10 Hz (Note, Section 4.4)

In [6]:
TF, H = 0.02, 0.1
fwd, bwd, tus = [0.0], [], []
for k in range(5):
    fwd.append(fwd[-1] + H / TF * (1 - fwd[-1]))
    bwd.append((TF * (bwd[-1] if bwd else 0.0) + H) / (TF + H))
    tus.append(((2 * TF - H) * (tus[-1] if tus else 0.0) + H * (1 + (1.0 if k else 0.0))) / (2 * TF + H))
print("gap factors: forward", 1 - H / TF, " backward", round(TF / (TF + H), 4), " Tustin", round((2 * TF - H) / (2 * TF + H), 4))
print("forward ", np.round(fwd[:5], 3))
print("backward", np.round(bwd[:4], 3))
print("Tustin  ", np.round(tus[:4], 3))
print("continuous filter at 0.1 s:", round(1 - np.exp(-H / TF), 3))

gap factors: forward -4.0  backward 0.1667  Tustin -0.4286
forward  [   0.    5.  -15.   65. -255.]
backward [0.833 0.972 0.995 0.999]
Tustin   [0.714 1.122 0.948 1.022]
continuous filter at 0.1 s: 0.993


## 6. The loop in z: the PI controller, the sampled robot and the poles (Note, Section 5)

In [7]:
for method, name in (("euler", "forward Euler"), ("backward_diff", "backward Euler"), ("bilinear", "Tustin")):
    num, den, _ = signal.cont2discrete(([KP, KI], [1, 0]), 0.1, method=method)
    print(f"PI at 10 Hz, {name:>14}: numerator {np.round(num.ravel(), 4)}, denominator {np.round(den, 4)}")
for Ts in (0.01, 0.1, 0.5):
    num, den, _ = signal.cont2discrete(([1], [M, B]), Ts, method="zoh")
    print(f"robot sampled every {Ts} s (zero-order hold): a = {-den[1]:.4f}, g = {num.ravel()[1]:.5f}")
a, g = np.exp(-B * 0.5 / M), (1 - np.exp(-B * 0.5 / M)) / B
print("check at steady state: 0.8825 x 0.5 + 0.05875 x 1 =", round(a * 0.5 + g * 1.0, 4))


def loop_poles(Ts):
    a = np.exp(-B * Ts / M)
    g = (1 - a) / B
    return np.roots([1, -(a - g * KP + 1), (a - g * KP) + g * KI * Ts])

for Ts in (0.5, 2.0):
    a, g = np.exp(-B * Ts / M), (1 - np.exp(-B * Ts / M)) / B
    a, g = round(a, 4), round(g, 5)                      # the rounded values the Note shows
    print(f"T = {Ts} s: a = {a}, g = {g}, g kp = {g * KP:.5f}, a - g kp = {a - g * KP:.5f}, g ki T = {g * KI * Ts:.5f},",
          f"last number = {a - g * KP + g * KI * Ts:.5f}, z coefficient = {-(a - g * KP + 1):.5f}")
s_cont = np.roots([M, B + KP, KI])
print("continuous poles", np.round(s_cont, 4), "rad/s")
for fs in (100, 10, 2, 1, 0.5):
    p = loop_poles(1 / fs)
    print(f"{fs:>5} Hz: poles {np.round(np.sort(p.real)[::-1], 4)}, farthest {abs(p).max():.4f};",
          f"continuous poles sent through e^(sT): {np.round(np.exp(s_cont / fs), 4)}")
Ts = np.arange(0.01, 3.0, 0.0005)
print("first unstable sample period:", round(Ts[np.argmax([abs(loop_poles(T)).max() > 1 + 1e-9 for T in Ts])], 4), "s")

PI at 10 Hz,  forward Euler: numerator [ 20. -19.], denominator [ 1. -1.]
PI at 10 Hz, backward Euler: numerator [ 21. -20.], denominator [ 1. -1.]
PI at 10 Hz,         Tustin: numerator [ 20.5 -19.5], denominator [ 1. -1.]
robot sampled every 0.01 s (zero-order hold): a = 0.9975, g = 0.00125
robot sampled every 0.1 s (zero-order hold): a = 0.9753, g = 0.01235
robot sampled every 0.5 s (zero-order hold): a = 0.8825, g = 0.05875
check at steady state: 0.8825 x 0.5 + 0.05875 x 1 = 0.5
T = 0.5 s: a = 0.8825, g = 0.05875, g kp = 1.17500, a - g kp = -0.29250, g ki T = 0.29375, last number = 0.00125, z coefficient = -0.70750
T = 2.0 s: a = 0.6065, g = 0.19673, g kp = 3.93460, a - g kp = -3.32810, g ki T = 3.93460, last number = 0.60650, z coefficient = 2.32810
continuous poles [-2.1754 -0.5746] rad/s
  100 Hz: poles [0.9943 0.9783], farthest 0.9943; continuous poles sent through e^(sT): [0.9785 0.9943]
   10 Hz: poles [0.9423 0.7861], farthest 0.9423; continuous poles sent through e^(sT): [0

first unstable sample period: 1.1175 s


## 7. Which rule in the loop (Note, Section 5.4)

In [8]:
for fs in (100, 10, 2):
    print(fs, "Hz:", {m: float(round(overshoot(sampled_loop(fs, m)), 1)) for m in ("fwd", "bwd", "tus")})

100 Hz: {'fwd': 5.0, 'bwd': 4.9, 'tus': 4.9}
10 Hz: {'fwd': 5.7, 'bwd': 5.1, 'tus': 5.4}
2 Hz: {'fwd': 17.5, 'bwd': 46.9, 'tus': 32.2}


## 8. Choosing the sample rate (Note, Section 6)

In [9]:
Tcl = signal.TransferFunction([KP, KI], [M, B + KP, KI])
mag = abs(signal.freqresp(Tcl, w)[1])
bw = w[np.argmax(mag < 1 / np.sqrt(2))]
print("closed-loop bandwidth", round(bw, 3), "rad/s =", round(bw / 2 / np.pi, 2), "Hz")
print("CTM rule: T <", round(1 / (30 * bw), 4), "s, so at least", round(30 * bw, 1), "samples per second")

closed-loop bandwidth 2.764 rad/s = 0.44 Hz
CTM rule: T < 0.0121 s, so at least 82.9 samples per second


## 9. Aliasing (Note, Section 7)

In [10]:
f = W_RIP / (2 * np.pi)
print("ripple", round(f, 4), "Hz; seen at 2 Hz as", round(abs(f - 2), 4), "Hz; turn per 2 Hz sample",
      round(np.degrees(W_RIP * 0.5), 1), "deg =", round(np.degrees(W_RIP * 0.5) - 360, 1), "deg")
for fs in (100, 10, 2):
    r = sampled_loop(fs, T=40, ripple=A_RIP, r0=0.5, r1=0.5)
    tail = r["v"][r["t"] > 20]
    print(f"{fs:>4} Hz: the true speed wobbles by {(tail.max() - tail.min()) / 2:.4f} m/s")
for tf in (0.2, 0.5):
    r = sampled_loop(2, T=40, ripple=A_RIP, tau_f=tf, r0=0.5, r1=0.5)
    tail = r["v"][r["t"] > 20]
    print(f"2 Hz with a {tf} s filter before the A/D: wobble {(tail.max() - tail.min()) / 2:.4f} m/s, "
          f"step overshoot {overshoot(sampled_loop(2, tau_f=tf, T=15)):.1f} percent")
r = sampled_loop(100, T=40, ripple=A_RIP, tau_f=0.005, r0=0.5, r1=0.5)
tail = r["v"][r["t"] > 20]
print(f"100 Hz with a 0.005 s filter: wobble {(tail.max() - tail.min()) / 2:.4f} m/s, "
      f"step overshoot {overshoot(sampled_loop(100, tau_f=0.005)):.1f} percent")

ripple 1.5915 Hz; seen at 2 Hz as 0.4085 Hz; turn per 2 Hz sample 286.5 deg = -73.5 deg
 100 Hz: the true speed wobbles by 0.0025 m/s


  10 Hz: the true speed wobbles by 0.0028 m/s


   2 Hz: the true speed wobbles by 0.0119 m/s


2 Hz with a 0.2 s filter before the A/D: wobble 0.0096 m/s, step overshoot 61.9 percent


2 Hz with a 0.5 s filter before the A/D: wobble 0.0036 m/s, step overshoot 98.8 percent


100 Hz with a 0.005 s filter: wobble 0.0025 m/s, step overshoot 5.0 percent
